In [ ]:
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader
from PIL import Image
from torchvision import transforms, models
import random 
import numpy as np
import torch

# Data pipeline
Raw Data -> Split -> Dataset -> Transform -> Collate -> DataLoader -> Model
- Augmentation changes input distribution
- Sampling changes class distributions inside minibatches
- Batch composition affects BatchNorm statistics
- Collate and resizing affect tensor shape and memory usage

# Data split
- Train: learn parameters
- Validation: tune hyperparameters and select checkpoints
- Test: final evaluation
- Plot histogram to check P(train) ~ P(test) (to have a look at the problem)

# Dataset

In [ ]:
# Core: __len__ and __getitem__ methods 
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Label
- Label have compability with loss
- Single label -> CrossEntropy
- Multi label -> BCEWithLogitsLoss
- Segmentation -> PixelwiseLoss

# Transform/Augmentation
- Must: preserve the semantic label
- Plot histogram for distribution shift detected
- Common:
+ Geometric: crop, flip, rotate, scale, perspective
+ Photometric: brightness, contrast, saturation, hue, blur
+ Regularization: MixUp, CutMix, Random Erasing
- Structured target:
+ Classification: Often image augmentation is enough, if lable unchanged
+ Detection/Segmentation: sync image and annotation transform
+ Bounding boxes
+ Masks
+ Keypoints
+ Polygons
- Train: random augmentation
- Validation/Test: deterministic preprocessing

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Pretrained Preprocessing

In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)

# Use official preprocessing recipe tied to the checkpoint
preprocess = weights.transforms()

# Collate
- Convert list of samples into one batch
- Default: stacks tensors with same shapes
- Custom collate when different image sizes, variable lengths or special handling for detection boxes/segmentation masks

In [ ]:
# Classification collate
def collate_fn(batch):
    xs, ys = zip(*batch)
    xs = torch.stack(xs)
    ys = torch.tensor(ys)
    return xs, ys

# Detection collate
def detection_collate_fn(batch):
    xs, ys = zip(*batch)
    return list(xs), list(ys)

# DataLoader

In [ ]:
loader = DataLoader(
    dataset=ImageDataset(image_paths=['path/to/image1.jpg', 'path/to/image2.jpg'], transform=train_tf),
    batch_size=32,
    shuffle=True,
    num_workers=4, # more workers = faster data loading
    pin_memory=True, # Faster host->device transfer
    collate_fn=collate_fn,
    persistent_workers=True # Reduce worker startup time
)

# Imbalance
- Weighted Sampler + Weighted Loss

In [ ]:
torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

labels = torch.tensor(train_labels)
class_count = torch.bincount(labels)
class_weights = 1. / class_count.float().clamp(min=1)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
criterion = class_weights.to(device) # Use class weights in loss function for better handling of class imbalance

# Reproducibility

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = seed + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    torch.cuda.manual_seed_all(worker_seed)

g = torch.Generator()
g.manual_seed(seed)